In [ ]:
import os
import pandas as pd
from getpass import getpass

from pydantic import BaseModel, Field
from typing import List
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate

print("Libraries loaded successfully!")

C:\Users\HP\AppData\Local\Temp\ipykernel_28432\598995206.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Libraries loaded successfully!


In [ ]:
load_dotenv()

In [2]:
## JOB DESCRIPTION  ###

job_description = """
We are hiring a Data Scientist.

Required Skills:
- Python
- SQL
- Pandas
- NumPy
- Machine Learning
- Scikit-learn
- Statistics
- Data Visualization
- Data Analysis

Preferred Skills:
- NLP
- Deep Learning
- TensorFlow
- PyTorch
- AWS or Cloud

Responsibilities:
- Clean and analyze data
- Build machine learning models
- Evaluate models
- Create data visualizations
- Find useful business insights
- Work with large datasets
"""
    
print(job_description)


We are hiring a Data Scientist.

Required Skills:
- Python
- SQL
- Pandas
- NumPy
- Machine Learning
- Scikit-learn
- Statistics
- Data Visualization
- Data Analysis

Preferred Skills:
- NLP
- Deep Learning
- TensorFlow
- PyTorch
- AWS or Cloud

Responsibilities:
- Clean and analyze data
- Build machine learning models
- Evaluate models
- Create data visualizations
- Find useful business insights
- Work with large datasets



In [3]:
## STRUCTURED OUTPUT  ###

class ResumeEvaluation(BaseModel):

    match_score: int = Field(
        description="Match score between 0 and 100"
    )

    candidate_summary: str

    matching_skills: List[str]

    missing_skills: List[str]

    strengths: List[str]

    weaknesses: List[str]

    hiring_recommendation: str

    justification: str
    

In [4]:
print("Structured output created!")

Structured output created!


In [5]:
## CREATE LLM  ###

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

structured_llm = llm.with_structured_output(
    ResumeEvaluation
)

print("GPT-4o-mini ready!")

GPT-4o-mini ready!


In [6]:
### CREATE PROMPT TEMPLATE  ###


prompt = ChatPromptTemplate.from_template("""
You are an AI Resume Screening Assistant.

IMPORTANT RULES:

1. Use ONLY information present in the RESUME CONTEXT.
2. Do not use outside knowledge about the candidate.
3. Do not invent skills, education, experience, projects or achievements.
4. Compare the resume with the Job Description.
5. If a required skill is not found in the resume, put it in Missing Skills.
6. The justification must be based only on resume evidence.
7. Match Score must be between 0 and 100.
8. If information is not available, write "Not found in resume".

JOB DESCRIPTION:

{job_description}


RESUME CONTEXT:

{context}
""")

print("Prompt created!")

Prompt created!


In [7]:
### CREATE LANGCHAIN CHAIN  ###

chain = prompt | structured_llm

print("LangChain RAG chain created!")

LangChain RAG chain created!


In [8]:
### LOAD PDF RESUME  ###

resume_path = "Resume_A.pdf"

loader = PyPDFLoader(resume_path)

documents = loader.load()

print("Number of pages:", len(documents))

Number of pages: 3


In [9]:
print(documents[0].page_content[:2000])

Meenal singh 
DATA SCIENTIST | MACHINE LEARNING | BUSINESS ANALYTICS 
    jaipur India |    +91-9259393961|       meenal_singh@gmail.com 
LinkedIn: linkedin.com/in/meenal | GitHub: github.com/meenal 
PROFILE 
Results-oriented Data Scientist with 1–3 years of experience applying statistical 
analysis, machine learning, and data visualization to business problems. Strong ability 
to work with large datasets, identify patterns, build predictive models, and 
communicate findings to technical and non-technical stakeholders. 
CORE COMPETENCIES 
Programming: Python, SQL 
Machine Learning: Supervised Learning, Unsupervised Learning, Ensemble Models, 
Model Validation 
Data Science: EDA, Feature Engineering, Statistical Analysis, Predictive Analytics 
Visualization: Power BI, Tableau, Matplotlib, Seaborn 
Databases: MySQL, PostgreSQL 
Libraries: Pandas, NumPy, Scikit-learn, XGBoost 
Development Tools: Git, GitHub, Jupyter Notebook 
Cloud: AWS Fundamentals 
PROFESSIONAL EXPERIENCE 
Junior Data S

In [10]:
###  TEXT SLITTER ###3

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(
    documents
)

print("Number of chunks:", len(chunks))

Number of chunks: 5


In [11]:
#### CREATE EMBEDDINGS  ###

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print("Embedding model ready!")

Embedding model ready!


In [12]:
### CREATE FAISS VECTOR DATABASE  ###

vector_db = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector database created!")

FAISS vector database created!


In [13]:
### RETRIEVER  ###

retriever = vector_db.as_retriever(
    search_kwargs={
        "k": 6
    }
)

print("Retriever created!")

Retriever created!


In [14]:
### RETRIEVE RESUME INFORMATION ###


retrieved_docs = retriever.invoke(
    job_description
)

print(
    "Retrieved chunks:",
    len(retrieved_docs)
)

Retrieved chunks: 5


In [15]:
#### CREATE CONTEXT  ###


context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

print(context)

Meenal singh 
DATA SCIENTIST | MACHINE LEARNING | BUSINESS ANALYTICS 
    jaipur India |    +91-9259393961|       meenal_singh@gmail.com 
LinkedIn: linkedin.com/in/meenal | GitHub: github.com/meenal 
PROFILE 
Results-oriented Data Scientist with 1–3 years of experience applying statistical 
analysis, machine learning, and data visualization to business problems. Strong ability 
to work with large datasets, identify patterns, build predictive models, and 
communicate findings to technical and non-technical stakeholders. 
CORE COMPETENCIES 
Programming: Python, SQL 
Machine Learning: Supervised Learning, Unsupervised Learning, Ensemble Models, 
Model Validation 
Data Science: EDA, Feature Engineering, Statistical Analysis, Predictive Analytics 
Visualization: Power BI, Tableau, Matplotlib, Seaborn 
Databases: MySQL, PostgreSQL 
Libraries: Pandas, NumPy, Scikit-learn, XGBoost 
Development Tools: Git, GitHub, Jupyter Notebook 
Cloud: AWS Fundamentals 
PROFESSIONAL EXPERIENCE

Libraries: Pa

In [16]:
###  GENERATE EVALUATION  ###


result = chain.invoke({
    "job_description": job_description,
    "context": context
})

print(result)

match_score=90 candidate_summary='Results-oriented Data Scientist with 1–3 years of experience in statistical analysis, machine learning, and data visualization. Proficient in Python, SQL, and various data science libraries, with a strong ability to work with large datasets and communicate findings effectively.' matching_skills=['Python', 'SQL', 'Pandas', 'NumPy', 'Machine Learning', 'Scikit-learn', 'Statistics', 'Data Visualization', 'Data Analysis'] missing_skills=['NLP', 'Deep Learning', 'TensorFlow', 'PyTorch', 'AWS or Cloud'] strengths=['Experience with large datasets', 'Strong statistical analysis skills', 'Proficient in data visualization tools like Power BI and Tableau', 'Experience in building predictive models', 'Ability to communicate findings to stakeholders'] weaknesses=['Limited experience with NLP and deep learning frameworks', 'No specific mention of TensorFlow or PyTorch expertise'] hiring_recommendation='Strongly recommend for hiring due to relevant skills and experie

In [17]:
#### DISPLAY MATCH SCORE  ###

print("Match Score:", result.match_score, "/100")

print("Matching Skills:")

for skill in result.matching_skills:
    print("-", skill)

Match Score: 90 /100
Matching Skills:
- Python
- SQL
- Pandas
- NumPy
- Machine Learning
- Scikit-learn
- Statistics
- Data Visualization
- Data Analysis


In [18]:
print("Missing Skills:")

for skill in result.missing_skills:
    print("-", skill)

Missing Skills:
- NLP
- Deep Learning
- TensorFlow
- PyTorch
- AWS or Cloud


In [19]:
print("Strengths:")

for strength in result.strengths:
    print("-", strength)

Strengths:
- Experience with large datasets
- Strong statistical analysis skills
- Proficient in data visualization tools like Power BI and Tableau
- Experience in building predictive models
- Ability to communicate findings to stakeholders


In [20]:
print("Weaknesses:")

for weakness in result.weaknesses:
    print("-", weakness)

Weaknesses:
- Limited experience with NLP and deep learning frameworks
- No specific mention of TensorFlow or PyTorch expertise


In [21]:
print("Hiring Recommendation:")

print(result.hiring_recommendation)

Hiring Recommendation:
Strongly recommend for hiring due to relevant skills and experience in data science and machine learning.


In [22]:
print("Justification:")

print(result.justification)

Justification:
The candidate possesses all the required skills listed in the job description, including Python, SQL, Pandas, NumPy, Machine Learning, Scikit-learn, Statistics, Data Visualization, and Data Analysis. They also have relevant experience in analyzing data, building models, and creating visualizations, which aligns well with the responsibilities outlined in the job description.


In [35]:
def evaluate_resume(pdf_file, job_description):

    # Step 1: Load PDF
    loader = PyPDFLoader(pdf_file)

    documents = loader.load()


    # Step 2: Split text
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150
    )

    chunks = splitter.split_documents(
        documents
    )


    # Step 3: Create embeddings
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small"
    )


    # Step 4: Create FAISS database
    vector_db = FAISS.from_documents(
        chunks,
        embeddings
    )


    # Step 5: Create retriever
    retriever = vector_db.as_retriever(
        search_kwargs={"k": 6}
    )


    # Step 6: Retrieve relevant information
    retrieved_docs = retriever.invoke(
        job_description
    )


    # Step 7: Create context
    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )


    # Step 8: Generate answer
    result = chain.invoke({
        "job_description": job_description,
        "context": context
    })


    return result

In [24]:
###

In [36]:
def show_result(res):
    print("Match Score:", res.match_score, "/100")
    print("Hiring Recommendation:", res.hiring_recommendation)
    
    print("\nMatching Skills:")
    for skill in res.matching_skills:
        print("-", skill)
        
    print("\nMissing Skills:")
    for skill in res.missing_skills:
        print("-", skill)
        
    print("\nJustification:")
    print(res.justification)

In [37]:
###  TEST CHECK  RESUME 1 ###

result_A = evaluate_resume(
    "E:\\data science\\gen ai\\Resume_A.pdf",
    job_description
)

show_result(result_A)

Match Score: 90 /100
Hiring Recommendation: Strongly recommend for hiring due to relevant skills and experience in data science and machine learning.

Matching Skills:
- Python
- SQL
- Pandas
- NumPy
- Machine Learning
- Scikit-learn
- Statistics
- Data Visualization
- Data Analysis

Missing Skills:
- NLP
- Deep Learning
- TensorFlow
- PyTorch
- AWS or Cloud

Justification:
The candidate possesses all the required skills listed in the job description, including Python, SQL, Pandas, NumPy, Machine Learning, Scikit-learn, Statistics, Data Visualization, and Data Analysis. They have relevant professional experience and projects that demonstrate their ability to clean and analyze data, build machine learning models, and create data visualizations. However, they lack experience in some preferred skills such as NLP and deep learning frameworks.


In [38]:
### TEST 2


result_B = evaluate_resume(
    "E:\\data science\\gen ai\\Resume_B.pdf",
    job_description
)

show_result(result_B)

Match Score: 95 /100
Hiring Recommendation: Strongly Recommend

Matching Skills:
- Python
- SQL
- Pandas
- NumPy
- Machine Learning
- Scikit-learn
- Statistics
- Data Visualization
- Data Analysis

Missing Skills:
- NLP
- Deep Learning
- TensorFlow
- PyTorch
- AWS or Cloud

Justification:
The candidate possesses all the required skills listed in the job description, including Python, SQL, Pandas, NumPy, Machine Learning, Scikit-learn, Statistics, Data Visualization, and Data Analysis. They also have relevant experience in developing machine learning models and working with large datasets. The candidate's education and certifications further support their qualifications for the role.


In [39]:
### TEST 3

result_C = evaluate_resume(
    "E:\\data science\\gen ai\\Resume_C.pdf",
    job_description
)

print("Missing Skills in Resume C:")

for skill in result_C.missing_skills:
    print("-", skill)

Missing Skills in Resume C:
- Pandas
- NumPy
- Machine Learning
- Scikit-learn
- Statistics
- Data Visualization
- Data Analysis
- NLP
- Deep Learning
- TensorFlow
- PyTorch
- AWS or Cloud


In [40]:
### TEST CASE  4 COMPARE A,B AND CW


results = {
    "Resume A": result_A,
    "Resume B": result_B,
    "Resume C": result_C
}

In [41]:
###  COMPARISON TABLE###

comparison = pd.DataFrame([
    {
        "Candidate": name,
        "Match Score": result.match_score,
        "Recommendation": result.hiring_recommendation,
        "Matching Skills": ", ".join(
            result.matching_skills
        ),
        "Missing Skills": ", ".join(
            result.missing_skills
        )
    }

    for name, result in results.items()
])

comparison

,Candidate,Match Score,Recommendation,Matching Skills,Missing Skills
0,Resume A,90,Strongly recommend for hiring due to relevant ...,"Python, SQL, Pandas, NumPy, Machine Learning, ...","NLP, Deep Learning, TensorFlow, PyTorch, AWS o..."
1,Resume B,95,Strongly Recommend,"Python, SQL, Pandas, NumPy, Machine Learning, ...","NLP, Deep Learning, TensorFlow, PyTorch, AWS o..."
2,Resume C,20,Not recommended for the Data Scientist position.,"Python, SQL","Pandas, NumPy, Machine Learning, Scikit-learn,..."


In [34]:
###  Hiring Recommendation with Justification

for name, result in results.items():

    print("\n" + "=" * 70)

    print("CANDIDATE:", name)

    print("MATCH SCORE:",
          result.match_score)

    print("RECOMMENDATION:",
          result.hiring_recommendation)

    print("JUSTIFICATION:")

    print(result.justification)


CANDIDATE: Resume A
MATCH SCORE: 90
RECOMMENDATION: Strongly recommend for hiring due to relevant experience and skills in data science and machine learning.
JUSTIFICATION:
The candidate possesses all the required skills listed in the job description, including Python, SQL, Pandas, NumPy, Machine Learning, Scikit-learn, Statistics, Data Visualization, and Data Analysis. They have relevant professional experience and projects that demonstrate their ability to clean and analyze data, build machine learning models, and create data visualizations. However, they lack some preferred skills such as NLP, Deep Learning, TensorFlow, and PyTorch.

CANDIDATE: Resume B
MATCH SCORE: 95
RECOMMENDATION: Highly recommended for hire due to strong alignment with required skills and relevant experience in data science and machine learning.
JUSTIFICATION:
The candidate possesses all the required skills listed in the job description, including Python, SQL, Pandas, NumPy, Machine Learning, Scikit-learn, Sta